# 05 - Hyperparameter Tuning

Uses Optuna to search for better XGBoost hyperparameters than Phase 4's defaults, and
compares three ways of handling class imbalance in the same search. Ends with the ONE
AND ONLY time this whole project touches the test set.

**This cell is slow -- expect it to take a while (up to `tuning.timeout_minutes` in
config.yaml) on the real dataset.** It's resumable: re-running the tuning cell picks up
from where a previous run left off (the study is saved to a sqlite file), so it's safe to
interrupt and come back to.

## What's new this phase

**Optuna / TPE sampler:** instead of guessing hyperparameters or trying every combination
in a grid, Optuna tries many combinations and learns from each attempt. The **TPE
(Tree-structured Parzen Estimator)** sampler builds a rough model of which regions of the
search space tend to score well and focuses future attempts there -- like tuning a radio
by noticing you're getting closer to a station, rather than blindly trying every frequency.

**The search space** (see `config.yaml: tuning.search_space`):
- `max_depth` -- how deep each tree can grow. Deeper = more complex patterns, but more
  risk of memorising noise.
- `learning_rate` -- how much each new tree corrects previous trees' mistakes.
- `n_estimators` -- how many trees to build.
- `min_child_weight` -- minimum data needed before a split is allowed (higher = more
  conservative).
- `subsample` / `colsample_bytree` -- what fraction of rows / columns each tree trains on.
- `gamma` -- minimum improvement a split must provide to be worth making.
- `reg_alpha` / `reg_lambda` -- L1/L2 regularisation, penalties against overly complex trees.
- `scale_pos_weight` -- extra weight given to the rare laundering class (only used by the
  `scale_pos_weight` imbalance method below, not a fixed setting).

**Pruning:** if a trial is clearly doing badly partway through, there's no point finishing
it. The median pruner cuts a trial once it's doing worse than the median of previous
trials at the same point, saving real time.

**Comparing imbalance handling:** rather than three separate 50-trial studies (one per
method -- up to 3x the runtime for the same answer), the imbalance method itself
(`none` / `scale_pos_weight` / `undersample`) is one of the things Optuna searches over,
within a single budget. The TPE sampler naturally spends more trials on whichever method
tends to score better.

**Why test is touched exactly once, at the end:** every time a decision gets adjusted
based on test performance, a little information leaks from test into that decision. Do it
repeatedly and the model ends up indirectly fit to test, the same way it can overfit to
train -- great score on this particular test set, worse on genuinely new data. So every
decision (features, model, hyperparameters, imbalance method, threshold) is made using only
train and validation. Test is used exactly once, after everything else is locked in.

In [ ]:
import sys
import os
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt
import optuna.visualization.matplotlib as ovm

from src.config import load_config
from src.train import load_feature_tables, TARGET_COL
from src.evaluate import save_metrics
from src.tune import run_study, summarize_imbalance_methods, retrain_best_and_evaluate_once

config = load_config(project_root / "config.yaml")
FIGURES_DIR = Path(config["paths"]["figures_dir"])
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Load train and validation

Test is NOT loaded here -- only right before the one-time final check, near the end.

In [ ]:
train_df, val_df = load_feature_tables(config)
print(f"train: {len(train_df):,} rows ({train_df[TARGET_COL].mean():.4%} laundering)")
print(f"val:   {len(val_df):,} rows ({val_df[TARGET_COL].mean():.4%} laundering)")

## Run the Optuna study

This is the slow cell. It runs up to `config['tuning']['n_trials']` trials or
`config['tuning']['timeout_minutes']` minutes, whichever comes first. Safe to interrupt --
re-running this cell resumes from the saved study rather than starting over.

In [ ]:
study = run_study(train_df, val_df, config)
print(f"\nCompleted {len(study.trials)} trials total.")
print(f"Best validation PR-AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

## Which imbalance method actually won?

In [ ]:
imbalance_comparison = summarize_imbalance_methods(study)
imbalance_comparison

## Optuna diagnostic plots

In [ ]:
ax = ovm.plot_optimization_history(study)
ax.figure.tight_layout()
ax.figure.savefig(FIGURES_DIR / "10_optuna_optimization_history.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
ax = ovm.plot_param_importances(study)
ax.figure.tight_layout()
ax.figure.savefig(FIGURES_DIR / "11_optuna_param_importance.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
ax = ovm.plot_parallel_coordinate(study)
ax.figure.tight_layout()
ax.figure.savefig(FIGURES_DIR / "12_optuna_parallel_coordinate.png", dpi=150, bbox_inches="tight")
plt.show()

## The one and only test-set evaluation

Everything above only used train and validation. This cell loads test for the first time
in the entire project, retrains with the best hyperparameters found, and reports test
metrics once. This cell should never be re-run with different hyperparameters chosen
afterward based on its result -- that would defeat the entire point.

In [ ]:
test_df = pd.read_parquet(Path(config["paths"]["features_dir"]) / "test_features.parquet")
print(f"test: {len(test_df):,} rows ({test_df[TARGET_COL].mean():.4%} laundering)")

test_metrics = retrain_best_and_evaluate_once(train_df, val_df, test_df, study.best_params, config)
test_metrics

## Before vs after: baseline vs tuned

Baseline numbers are Phase 4's XGBoost (default), evaluated on VALIDATION.
Tuned numbers are this phase's best model, evaluated on TEST (the numbers aren't
perfectly apples-to-apples for that reason, but TEST is what actually matters for an
honest final number -- see the leakage discussion above).

In [ ]:
import json
baseline_path = Path(config["paths"]["metrics_dir"]) / "baseline.json"
baseline_metrics = json.loads(baseline_path.read_text())["xgboost_default"]

before_after = pd.DataFrame({
    "baseline (val)": baseline_metrics,
    "tuned (test)": test_metrics,
}).T[["pr_auc", "recall", "precision", "f1", "business_cost_usd"]]
before_after

## Save results

In [ ]:
save_metrics({"xgboost_tuned_test": test_metrics, "best_params": study.best_params}, config, filename="tuned.json")
print(f"Saved to {Path(config['paths']['metrics_dir']) / 'tuned.json'}")